# Reproduce Binding Site Prediction on Temporal Set
Here we provide a reproducibility script to predict binding sites on the temporal set using the SMARTBind model. The script loads the pre-trained SMARTBind models, predict binding sites for a given RNA-ligand pair, and averages the predictions across multiple models, and finally calculate AUC scores with respect to ground truth binding sites.

In [1]:
import warnings

import pandas as pd
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

import sys
sys.path.append("../../../../..") 
import pickle
import logging

from smartbind.preprocess import convert_smiles_to_pf2
from smartbind import BindingPL
import os

for logger_name in [
    "lightning_fabric.utilities.seed",
    "pytorch_lightning.utilities.seed",
    "lightning.pytorch.utilities.seed",
]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

In [2]:
def predict_binding(target_rna_chain, ligand_fp2, best_model_paths):
    binding_site_predictions_dict = {i: [] for i in range(len(target_rna_chain))}
    for weight_name in os.listdir(best_model_paths):
        weight_path = os.path.join(best_model_paths, weight_name)
        smartbind_binding_model = BindingPL(device='cpu', vs_mode=True)
        smartbind_binding_model.load_pretrained_model(model_path=weight_path, device='cpu', mode='inference')
        binding_prediction = smartbind_binding_model.predict_binding(target_rna_chain, ligand_fp2)
        binding_prediction = binding_prediction.detach().numpy()
        normalized_binding_prediction = (binding_prediction - min(binding_prediction)) / (max(binding_prediction) - min(binding_prediction)) 
        for i in range(len(target_rna_chain)):
            binding_site_predictions_dict[i].append(normalized_binding_prediction[i])

    average_predictions = []
    for i in range(len(target_rna_chain)):
        average_predictions.append(sum(binding_site_predictions_dict[i]) / 10)

    normalized_binding_site_prediction = ((average_predictions - min(average_predictions)) /
                                        (max(average_predictions) - min(average_predictions)))
    normalized_binding_site_prediction = [round(i, 3) for i in normalized_binding_site_prediction]
    normalized_binding_site_prediction = [float(i) for i in normalized_binding_site_prediction]
    return normalized_binding_site_prediction

In [3]:
result_list = {}
raw_result = {}
gt_pd = pd.read_csv('temporal_test_binding_groundtruth.csv')
for i in range(len(gt_pd)):
    pdbId = gt_pd.loc[i, 'PDB ID']
    print(f"Processing {i+1}/{len(gt_pd)}: PDB ID {pdbId}")
    ligandId = gt_pd.loc[i, 'Ligand ID']
    rna_sequence = gt_pd.loc[i, 'RNA sequence']
    ligand_smiles = gt_pd.loc[i, 'SMILES']
    binding_gt = gt_pd.loc[i, 'binding_groundtruth']
    normalized_predict = predict_binding(rna_sequence, convert_smiles_to_pf2(ligand_smiles), '../../../../../SMARTBind_weight')
    normalized_predict = [round(i, 3) for i in normalized_predict]
    gt_list = [int(i) for i in binding_gt]
    raw_result[f'{pdbId}_{ligandId}'] = {
        'groundtruth': gt_list,
        'prediction': normalized_predict
    }
    auc_score = roc_auc_score(gt_list, normalized_predict)
    print(f"PDB ID: {pdbId}, Ligand ID: {ligandId}, AUC Score: {auc_score:.4f}")
    result_list[f'{pdbId}_{ligandId}'] = auc_score
# save raw results
with open('SMARTBind_binding_site_inference_results.pkl', 'wb') as f:
    pickle.dump(raw_result, f)

Processing 1/34: PDB ID 9LKU
PDB ID: 9LKU, Ligand ID: GMP, AUC Score: 0.8878
Processing 2/34: PDB ID 9LKW
PDB ID: 9LKW, Ligand ID: GUN, AUC Score: 0.8922
Processing 3/34: PDB ID 9LKC
PDB ID: 9LKC, Ligand ID: GNG, AUC Score: 0.8154
Processing 4/34: PDB ID 9LKF
PDB ID: 9LKF, Ligand ID: GMP, AUC Score: 0.8029
Processing 5/34: PDB ID 9V4X
PDB ID: 9V4X, Ligand ID: 9QC, AUC Score: 0.8667
Processing 6/34: PDB ID 9V4Y
PDB ID: 9V4Y, Ligand ID: ANG, AUC Score: 0.8062
Processing 7/34: PDB ID 9HRO
PDB ID: 9HRO, Ligand ID: TOY, AUC Score: 0.2386
Processing 8/34: PDB ID 9IWF
PDB ID: 9IWF, Ligand ID: XAN, AUC Score: 0.8423
Processing 9/34: PDB ID 9I9W
PDB ID: 9I9W, Ligand ID: B2R, AUC Score: 0.5208
Processing 10/34: PDB ID 8ZNQ
PDB ID: 8ZNQ, Ligand ID: NAZ, AUC Score: 0.6100
Processing 11/34: PDB ID 9V4U
PDB ID: 9V4U, Ligand ID: NPR, AUC Score: 0.7945
Processing 12/34: PDB ID 9LJN
PDB ID: 9LJN, Ligand ID: GUN, AUC Score: 0.8495
Processing 13/34: PDB ID 9V50
PDB ID: 9V50, Ligand ID: A1LXN, AUC Score: 

In [6]:
result_list

{'9LKU_GMP': 0.8877777777777779,
 '9LKW_GUN': 0.8922222222222221,
 '9LKC_GNG': 0.8153988868274582,
 '9LKF_GMP': 0.8028756957328386,
 '9V4X_9QC': 0.8666666666666667,
 '9V4Y_ANG': 0.8061904761904761,
 '9HRO_TOY': 0.23863636363636365,
 '9IWF_XAN': 0.8422619047619048,
 '9I9W_B2R': 0.5208333333333334,
 '8ZNQ_NAZ': 0.6100478468899522,
 '9V4U_NPR': 0.7945269016697587,
 '9LJN_GUN': 0.8495098039215686,
 '9V50_A1LXN': 0.803921568627451,
 '9LKE_HPA': 0.8300189393939393,
 '9IWG_XAN': 0.8279773156899811,
 '9EC4_CTI': 0.668724279835391,
 '9EBP_LUM': 0.6935714285714287,
 '9LKV_GNG': 0.870754716981132,
 '8K7W_2ZY': 0.8043478260869565,
 '8T5O_TAC': 0.5251829530779165,
 '9MQS_5GP': 0.6363691560412872,
 '9MQT_A1BNU': 0.5665584415584416,
 '8ZAU_GUN': 0.7782738095238095,
 '8KED_GUN': 0.9071499503475671,
 '8KHH_GMP': 0.9357142857142858,
 '8KEB_GNG': 0.9046673286991063,
 '9FN2_SAH': 0.8353365384615385,
 '9IOS_53D': 0.5642857142857143,
 '9IO1_53D': 0.5641025641025641,
 '9IOR_53D': 0.625,
 '9IOU_53D': 0.929487

In [7]:
# Calculate mean
mean_auc = sum(result_list.values()) / len(result_list)
print(f"Mean AUC Score across all RNA-ligand pairs: {mean_auc:.4f}")

Mean AUC Score across all RNA-ligand pairs: 0.7451
